## Geometric features extraction and classifier testing

In [ ]:
import re
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, classification_report, ConfusionMatrixDisplay)

FOLDER = Path("../data/dataset_squat")
CONDITIONS = ["pre", "per", "post"]
SEED = 42 
Path("figures").mkdir(exist_ok=True)

In [ ]:
# All images
files = pd.DataFrame([
    {"path": str(path), "condition": condition}
    for condition in CONDITIONS
    for path in sorted((FOLDER / f"squat_{condition}").glob("*.jpg"))
])

# Random stratified split 60/20/20
trainval, test = train_test_split(
    files, test_size=0.20, stratify=files["condition"], random_state=SEED)
train, val = train_test_split(
    trainval, test_size=0.25, stratify=trainval["condition"], random_state=SEED)

# No overlap between train, validation and test sets
assert not (set(train["path"]) & set(val["path"]))
assert not (set(train["path"]) & set(test["path"]))
assert not (set(val["path"]) & set(test["path"]))

splits = pd.concat([train.assign(set="train"), val.assign(set="val"), test.assign(set="test")])
splits.to_csv("splits.csv", index=False)

print(pd.crosstab(splits["set"], splits["condition"]).loc[["train", "val", "test"], CONDITIONS])

In [ ]:
# Viola-Jones face detection algorithm
_CASCADE = cv2.CascadeClassifier(
    cv2.data.haarcascades + "haarcascade_frontalface_default.xml"
)
if _CASCADE.empty():
    raise RuntimeError("Cascade non chargée : vérifie l'installation d'OpenCV (<5)")


def face_detect(image, scale_factor=1.1, min_neighbors=5,
                    min_size=(60, 60), roi_ratio=1.0):
    """
    Detect face (Viola-Jones) and return the cropped BGR image.
    image     : path (str / Path) or BGR table
    roi_ratio : central fraction of the bounding box kept (0.6 = centre du visage)
    Returns (face_bgr, bbox) or (None, None) if no face is found.
    """
    img = cv2.imread(str(image)) if isinstance(image, (str, Path)) else image
    if img is None:
        raise FileNotFoundError(f"Unable to read image: {image}")

    grey = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    faces = _CASCADE.detectMultiScale(
        grey, scaleFactor=scale_factor,
        minNeighbors=min_neighbors, minSize=min_size,
    )
    if len(faces) == 0:
        return None, None

    x, y, w, h = max(faces, key=lambda b: b[2] * b[3])                  # we keep the largest detected face
    dx, dy = int(w * (1 - roi_ratio) / 2), int(h * (1 - roi_ratio) / 2) # central crop
    face = img[y + dy : y + h - dy, x + dx : x + w - dx]
    return face, (int(x), int(y), int(w), int(h))

In [ ]:
META = ["path", "condition", "set"]
EXCLUDE = ()
FEATURES = [c for c in data.columns if c not in META and not c.startswith(EXCLUDE)]

d_train, d_val, d_test = (data[data["set"] == j] for j in ("train", "val", "test"))
X_train, y_train = d_train[FEATURES], d_train["condition"]
X_val,   y_val   = d_val[FEATURES],   d_val["condition"]
X_test,  y_test  = d_test[FEATURES],  d_test["condition"]
print(f"{len(FEATURES)} features used")

models = {
    "Baseline (stratified)": DummyClassifier(strategy="stratified", random_state=SEED),
    "KNN": make_pipeline(
        StandardScaler(), KNeighborsClassifier(n_neighbors=5, weights="distance", n_jobs=-1)),
    "Logistic Regression": make_pipeline(
        StandardScaler(), LogisticRegression(max_iter=2000, class_weight="balanced", random_state=SEED, n_jobs=-1)),
    "Random Forest": RandomForestClassifier(
        n_estimators=300, class_weight="balanced", random_state=SEED, n_jobs=-1),
}

# Comparison on validation set
scores = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_val)
    scores[name] = {"accuracy": accuracy_score(y_val, pred),
                   "balanced_accuracy": balanced_accuracy_score(y_val, pred)}

scores_df = pd.DataFrame(scores).T.round(3)
print(scores_df)

best_model = max(scores, key=lambda n: scores[n]["balanced_accuracy"])
print("Best model on validation :", best_model)

# Final model trained on train + val
final_model = models[best_model]
final_model.fit(pd.concat([X_train, X_val]), pd.concat([y_train, y_val]))

In [ ]:
# Final evaluation on test
pred_test = final_model.predict(X_test)

print(f"Model: {best_model}")
print(f"Accuracy          : {accuracy_score(y_test, pred_test):.3f}")
print(f"Balanced accuracy : {balanced_accuracy_score(y_test, pred_test):.3f}\n")
print(classification_report(y_test, pred_test, labels=CONDITIONS, zero_division=0))

# Confusion matrix
fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay.from_predictions(
    y_test, pred_test, labels=CONDITIONS, cmap="Blues", ax=ax)
ax.set_title(f"Confusion Matrix (test)\n{best_model}")
plt.tight_layout()
plt.savefig("figures/confusion_geometry_baseline.png", dpi=150, bbox_inches="tight")
plt.show()

# Feature importance 
rf = models["Random Forest"]
imp = pd.Series(rf.feature_importances_, index=FEATURES).sort_values().tail(10)
imp.plot.barh(figsize=(6, 4), title="Top 10 features (Random Forest)")
plt.tight_layout()
plt.savefig("figures/importance_features_baseline.png", dpi=150, bbox_inches="tight")
plt.show()